# Exercise 01

## 1 Loading the Dataset

In [992]:
from sklearn.datasets import load_digits
import numpy as np

digits = load_digits()
print(digits.keys())
data = digits["data"]
print(data.shape)
images = digits["images"]
target = digits["target"]
target_names = digits["target_names"]
print(data.dtype)

dict_keys(['data', 'target', 'frame', 'feature_names', 'target_names', 'images', 'DESCR'])
(1797, 64)
float64


In [993]:
indices = np.where((target == 3) | (target == 8))   # find indices where target is 8 or 3
X = data[indices]
X = np.concatenate((X, np.ones((X.shape[0], 1))
                    ), axis=1)
y = target[indices]
y = np.where(y == 3, 1, -1)   # 3 => 1, 8 => -1


### 1.1 Classification with sklearn

In [994]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

clf1 = LogisticRegression(random_state=0, C=8, max_iter=200)
scores1 = cross_val_score(clf1, X_train, y_train, cv=5)
print(np.average(scores1))

clf2 = LogisticRegression(random_state=0, C=1, max_iter=200)
scores2 = cross_val_score(clf2, X_train, y_train, cv=5)
print(np.average(scores2))

clf5 = LogisticRegression(random_state=0, C=0.2, max_iter=200)
scores5 = cross_val_score(clf5, X_train, y_train, cv=5)
print(np.average(scores5))

clf6 = LogisticRegression(random_state=0, C=0.01, max_iter=200)
scores6 = cross_val_score(clf6, X_train, y_train, cv=5)
print(np.average(scores6))

# => best result with C=1
regularization = 1


0.9905869324473976
0.9905869324473976
0.9859357696566999
0.9859357696566999


## 1.2 Optimization Methods

In [995]:
def sigmoid(z): return 1 / (1 + np.exp(-z))

$\frac{\partial \text{ loss }}{\partial \beta} = \beta^T + \frac{\lambda}{N} \sum^N_{i=1} \text{sigmoid}(-y_i x_i \beta)(-y_i x_i)$

In [996]:
def gradient(beta, X, y, regularization):
    N = y.shape[0]
    sum = 0
    for i in range(N):
        sum += sigmoid(-y[i] * X[i].dot(beta)) * (-y[i] * X[i])
    return beta.T + (regularization / N) * sum


In [997]:
def predict(beta, X):
    p = sigmoid(X.dot(beta))
    return np.where(p > 0.5, 1, -1)

In [998]:
def zero_one_loss(y_prediction, y_truth):
   return np.sum(np.where((y_prediction * y_truth) == -1)) / np.shape(y_prediction[0])

#### Gradient Descent

In [999]:
def gradient_descent(m, X, y, beta_initial, learning_rate, regularization):
    beta = beta_initial
    for iter in range(m):
        beta = beta - learning_rate * gradient(beta, X, y, regularization)
    return beta

In [1000]:
beta = gradient_descent(1000, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate=0.001, regularization=1)


In [1001]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])

err 0.006993006993006993


#### Stochastic Gradient Descent

In [1002]:
def get_learning_rate(step, intitial, decrease_factor):
    return intitial / (1 + decrease_factor * step)


In [1003]:
# with replacement, as after about 200 interations no data points would be left for training
def stochastic_gradient_descent(m, X, y, beta_initial, learning_rate_initial, learning_rate_decrease_factor, regularization):
    beta = beta_initial
    learning_rate = learning_rate_initial

    samples = np.random.choice(y.shape[0], m, True)

    for cnt in range(m):
        index = samples[cnt]
        beta = beta - learning_rate * gradient(beta, X[index][np.newaxis, :], np.array([y[index]]), regularization)
        learning_rate = get_learning_rate(
            cnt, learning_rate_initial, learning_rate_decrease_factor)
    return beta

In [1004]:
beta = stochastic_gradient_descent(1000, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate_initial=0.1, learning_rate_decrease_factor = 2, regularization=1)


In [1005]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.1258741258741259


#### SG Minibatch

In [1006]:
# with replacement, as after about 200 interations no data points would be left for training
def minibatch(m, B, X, y, beta_initial, learning_rate_initial, learning_rate_decrease_factor, regularization):
    beta = beta_initial
    learning_rate = learning_rate_initial
    for cnt in range(m):
        samples = np.random.choice(y.shape[0], B, True)
        beta = beta - learning_rate * \
            gradient(beta, X[samples], y[samples], regularization)
        learning_rate = get_learning_rate(
            cnt, learning_rate_initial, learning_rate_decrease_factor)
    return beta


In [1007]:
beta = minibatch(1000, 50, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate_initial=0.1, learning_rate_decrease_factor=2, regularization=1)


In [1008]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.03496503496503497


#### SG Momentum

In [1009]:
# with replacement, as after about 200 interations no data points would be left for training
def momentum(m, X, y, beta_initial, learning_rate_initial, learning_rate_decrease_factor, regularization, momentum, g_initial):
    beta = beta_initial
    g = g_initial
    learning_rate = learning_rate_initial

    samples = np.random.choice(y.shape[0], m, True)

    for cnt in range(m):
        index = samples[cnt]
        g = momentum * g + (1 - momentum) * gradient(beta, X[index][np.newaxis, :], np.array([y[index]]), regularization)
        beta = beta - learning_rate * g
        learning_rate = get_learning_rate(
            cnt, learning_rate_initial, learning_rate_decrease_factor)
    return beta


In [1010]:
beta = momentum(1000, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate_initial=0.1, learning_rate_decrease_factor=2, regularization=1, momentum=0.99, g_initial=np.zeros(
    (X_train.shape[1])))


In [1011]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.013986013986013986


#### ADAM

In [1012]:
# with replacement, as after about 200 interations no data points would be left for training
def adam(m, X, y, beta_initial, learning_rate, regularization, momentum1, momentum2, g_initial, q_initial, epsilon):
    beta = beta_initial
    g = g_initial
    q = q_initial

    samples = np.random.choice(y.shape[0], m, True)

    for cnt in range(m):
        index = samples[cnt]

        l = gradient(beta, X[index][np.newaxis, :], np.array([y[index]]), regularization)

        g = momentum1 * g + (1 - momentum1) * l
        q = momentum2 * q + (1 - momentum2) * (l**2)


        g_ = g / (1 - (momentum1**(cnt+1)))
        q_ = q / (1 - (momentum2**(cnt+1))) 

        beta = beta - (learning_rate * g_) / (np.sqrt(q_) + epsilon)
    return beta


In [1013]:
beta = adam(1000, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate=0.0001, regularization=1, momentum1=0.9, momentum2=0.999, g_initial=np.zeros(
    (X_train.shape[1])), q_initial=np.zeros((X_train.shape[1])), epsilon=0.00000001)


In [1014]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.027972027972027972


#### Stochastic Average Gradient

In [1015]:
# with replacement, as after about 200 interations no data points would be left for training
def stoachstic_average_gradient(m, X, y, beta_initial, learning_rate, regularization):

    N = y.shape[0]

    g_stored = np.array([-y[i] * X[i].T * sigmoid(-y[i] * X[i].dot(beta_initial)) for i in range(y.shape[0])])

    g = (1 / N) * np.sum(g_stored, axis=0)

    beta = beta_initial

    samples = np.random.choice(y.shape[0], m, True)

    for cnt in range(m):
        index = samples[cnt]


        g_ = (-y[index] * X[index].T) * sigmoid(-y[index] * X[index].dot(beta))

        g = g + (1/N) * (g_ - g_stored[index])


        g_stored[index] = g_


        beta = beta * (1 - (learning_rate) / regularization) - learning_rate * g


    return beta


In [1016]:
beta = stoachstic_average_gradient(1000, X_train, y_train, beta_initial=np.zeros(
    (X_train.shape[1])), learning_rate=0.0001, regularization=1)


In [1017]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.013986013986013986


#### Dual Coordinate Ascent

In [1074]:
# with replacement, as after about 200 interations no data points would be left for training
def stoachstic_average_gradient(m, X, y, learning_rate, regularization, epsilon):

    N = y.shape[0]

    alpha = np.random.uniform(0, 1, N)

    beta = (regularization / N) * np.sum(np.array([alpha[i] * y[i] * X[i].T for i in range(N)]), axis=0)

    samples = np.random.choice(y.shape[0], m, True)

    for cnt in range(m):
        index = samples[cnt]

        f1 = y[index] * X[index].dot(beta) + np.log(alpha[index] / (1 - alpha[index]))
        f2 = (regularization / N) * (X[index].dot(X[index].T)) + 1 / (alpha[index] * (1 - alpha[index]))

        alpha_old = alpha

        alpha[index] = np.clip(alpha[index] - f1 / f2, epsilon, 1 - epsilon)

        beta = beta + (regularization / N) * y[index] * X[index].T * (alpha[index] - alpha_old[index])

    return beta


In [1079]:
beta = stoachstic_average_gradient(
    1000, X_train, y_train, learning_rate=0.0001, regularization=1, epsilon=0.001)


In [1080]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.11888111888111888


#### Newton/Raphson

In [1139]:
def newton(m, X, y, learning_rate, regularization, epsilon, W_initial, beta_initial):

    beta = beta_initial
    
    N = y.shape[0]

    W = W_initial

    samples = np.random.choice(y.shape[0], m, True)


    for cnt in range(m):

        index = samples[cnt]


        z = X.dot(beta)

        y_ = y / sigmoid(y * z)

        W = np.diag([(regularization / N) * sigmoid(z[i]) * sigmoid(-z[i]) for i in range(N)])

        beta = np.linalg.inv(np.identity(X.shape[1]) + X.T.dot(W.dot(X))).dot(X.T.dot(W.dot(z + y_)))

    return beta


In [1140]:
beta = newton(
    1000, X_train, y_train, learning_rate=0.0001, regularization=1, epsilon=0.001, W_initial=np.zeros((y.shape[0], y.shape[0])), beta_initial=np.zeros(X.shape[1]))


In [1141]:
# Testing performance
y_pred = predict(beta, X_test)
err = (y_pred * y_test)
err_sum = np.sum(np.where(err < 0, 1, 0))
print("err", err_sum / X_test.shape[0])


err 0.006993006993006993
